# Public release note

This is a sanitized public copy of `omniASR_20_K3_KenLM_V6_build_eval(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# 20.K3 — Construction et évaluation des KenLM V6

Ce notebook autonome consomme le corpus `20.K2` déjà validé et construit **5 candidats KenLM par langue** : ordres 3, 4 et 5 complets, puis deux 5-grammes avec pruning conservateur.

Il effectue ensuite une comparaison intrinsèque sans fuite (perplexité, OOV, taille et RSS dans un processus propre), conserve au plus **2 V6 par langue**, et laisse les V4/V5 historiques intacts pour la comparaison WER de `20.K4`.

## Exécution

1. Utiliser de préférence un runtime **CPU avec RAM augmentée**. Un A100 peut exécuter la cellule mais ne sera pas utilisé.
2. Vérifier qu'il reste au moins **25 Gio sur `/content`** et **16 Gio sur Drive** pendant la construction.
3. Exécuter l'unique cellule de code ci-dessous.
4. En cas de déconnexion, remonter Drive et relancer exactement la même cellule : chaque candidat complet est vérifié par SHA puis réutilisé.

Durée indicative : **1 à 2 h 30**. Un message de progression apparaît au moins chaque minute pendant les longues commandes KenLM.

Sortie attendue : `STATUT 20.K3 : PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING`.

Le notebook ne modifie ni `kenlm_tuning_by_domain_v3.json`, ni le runtime actif, ni un fichier de soumission Kaggle.

In [ ]:
# 20.K3 — Construction et evaluation intrinsèque des KenLM V6
#
# Cette cellule consomme exclusivement le corpus textuel 20.K2 fige. Elle
# compile une revision precise de KenLM, construit plusieurs candidats par
# langue, reconstruit le held-out de 20.K2 sans en persister le texte, mesure
# perplexite/taux OOV/RSS, puis conserve au plus deux V6 par langue pour 20.K4.
# Elle ne modifie ni les KenLM historiques, ni le runtime actif, ni une
# soumission Kaggle.

from __future__ import annotations

import gc
import gzip
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable


# ---------------------------------------------------------------------------
# 0. Dependances Python et Drive
# ---------------------------------------------------------------------------

REQUIRED_MODULES = {
    "pyarrow": "pyarrow>=14.0.0",
    "pandas": "pandas>=2.0.0",
    "ftfy": "ftfy>=6.2.0",
    "regex": "regex>=2023.12.25",
}

missing = [
    requirement
    for module, requirement in REQUIRED_MODULES.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    print("Installation des dependances Python 20.K3 :", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        check=True,
    )

import ftfy
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import regex as uregex
from ftfy.badness import badness as ftfy_badness

try:
    from google.colab import drive

    if not Path("/content/persistent_storage").exists():
        drive.mount("/content/drive")
except ImportError:
    pass


# ---------------------------------------------------------------------------
# 1. Constantes immuables et grille de candidats
# ---------------------------------------------------------------------------

FT_ROOT = Path(
    os.environ.get(
        "AFRIVOICES_FT_ROOT",
        "/content/afrivoices_project/"
        "finetune_runs/experiment_run",
    )
)
PROJECT_ROOT = FT_ROOT.parent.parent
K1_REPORT_ROOT = FT_ROOT / "reports/kenlm_v6_20_K1"
K2_REPORT_ROOT = FT_ROOT / "reports/kenlm_v6_20_K2"
K3_REPORT_ROOT = FT_ROOT / "reports/kenlm_v6_20_K3"
K3_MODEL_ROOT = FT_ROOT / "kenlm_models_v6_20_K3"

EXPECTED_K1_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K1_REPORT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
EXPECTED_K2_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K2_CONTRACT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)
KENLM_REPOSITORY = "https://github.com/kpu/kenlm.git"
KENLM_COMMIT = "4cb443e60b7bf2c0ddf3c745378f76cb59e254e5"

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
LANGUAGE_ALIASES = {
    "sw": {"sw", "swa", "swh", "swahili", "kiswahili"},
    "kik": {"kik", "ki", "kikuyu", "gikuyu"},
    "kln": {"kln", "sgc", "niq", "kalenjin", "kipsigis", "nandi"},
    "luo": {"luo", "dholuo"},
    "som": {"som", "so", "somali", "soomali", "af-soomaali"},
    "mas": {"mas", "maasai", "maa", "saq"},
}
TEXT_PRIORITY = (
    "text_norm",
    "text",
    "transcription",
    "transcript",
    "actualSentence",
    "sentence",
    "raw_transcription",
)
AUDIO_COLUMN_PATTERNS = (
    "audio",
    "waveform",
    "samples",
    "sample_rate",
    "sampling_rate",
    "speech_array",
)

NORMALIZER_VERSION = "afrivoices-kenlm-v6-nfc-ftfy-safe-20260715"
PIPELINE_REVISION = "afrivoices-kenlm-v6-20k3-r3-20260715"
MAX_WORDS_PER_LINE = 80
AUDIT_MODULUS = 5
AUDIT_RESIDUE = 0
LMPLZ_MEMORY = "35%"
MAX_BINARY_BYTES = 512 * 2**20
MAX_QUERY_RSS_BYTES = 1 * 2**30
FINAL_EDGE_RSS_BYTES = 8 * 2**30
FINAL_EDGE_SAFETY_MARGIN = 1.15
PPL_TOLERANCE_RATIO = 1.01
PPL_TOLERANCE_BITS = math.log2(PPL_TOLERANCE_RATIO)
AUDIT_TOLERANCE_BITS = 0.02
MIN_LOCAL_FREE_BYTES = 25 * 2**30
MIN_DRIVE_FREE_BYTES = 16 * 2**30

CANDIDATE_GRID = (
    {"name": "o3_full", "order": 3, "prune": []},
    {"name": "o4_full", "order": 4, "prune": []},
    {"name": "o5_full", "order": 5, "prune": []},
    {"name": "o5_p1", "order": 5, "prune": [0, 0, 1]},
    {"name": "o5_p2", "order": 5, "prune": [0, 0, 2]},
)

HISTORICAL_MODELS = {
    "sw": {"family": "v4", "path": FT_ROOT / "kenlm_models_v4/sw.binary"},
    "kik": {"family": "v4", "path": FT_ROOT / "kenlm_models_v4/kik.binary"},
    "kln": {"family": "v5", "path": FT_ROOT / "kenlm_models_v5/kln.binary"},
    "luo": {"family": "v5", "path": FT_ROOT / "kenlm_models_v5/luo.binary"},
    "som": {"family": "v5", "path": FT_ROOT / "kenlm_models_v5/som.binary"},
    "mas": {"family": "v5", "path": FT_ROOT / "kenlm_models_v5/mas.binary"},
}


# ---------------------------------------------------------------------------
# 2. Helpers stricts de provenance/publication
# ---------------------------------------------------------------------------

def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path, block_size: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value: Any, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
    os.replace(temporary, path)
    print("csv  ->", path)


def read_small_json(path: Path, maximum_bytes: int = 64 << 20) -> dict[str, Any]:
    path = Path(path)
    assert path.is_file() and not path.is_symlink(), path
    assert 0 < path.stat().st_size <= maximum_bytes, path
    value = json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(value, dict), path
    return value


def require_child(path: Path | str, parent: Path | str) -> Path:
    path = Path(path)
    parent = Path(parent)
    assert path.exists(), path
    resolved_path = path.resolve()
    resolved_parent = parent.resolve()
    assert os.path.commonpath([str(resolved_path), str(resolved_parent)]) == str(
        resolved_parent
    ), (path, parent)
    return resolved_path


def artifact_metadata(path: Path) -> dict[str, Any]:
    path = Path(path)
    assert path.is_file() and not path.is_symlink() and path.stat().st_size > 0
    return {"bytes": path.stat().st_size, "sha256": sha256_file(path)}


def publish_file(source: Path, target: Path) -> None:
    source = Path(source)
    target = Path(target)
    assert source.is_file() and source.stat().st_size > 0, source
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + f".tmp-{os.getpid()}")
    if temporary.exists():
        temporary.unlink()
    with source.open("rb") as input_handle, temporary.open("wb") as output_handle:
        shutil.copyfileobj(input_handle, output_handle, length=16 << 20)
        output_handle.flush()
        os.fsync(output_handle.fileno())
    assert temporary.stat().st_size == source.stat().st_size
    assert sha256_file(temporary) == sha256_file(source)
    os.replace(temporary, target)


def copy_verified(source: Path, target: Path, expected_sha256: str) -> None:
    source = Path(source)
    target = Path(target)
    if target.is_file() and sha256_file(target) == expected_sha256:
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + f".tmp-{os.getpid()}")
    if temporary.exists():
        temporary.unlink()
    shutil.copy2(source, temporary)
    assert sha256_file(temporary) == expected_sha256
    os.replace(temporary, target)


def run_checked(
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    timeout: int | None = None,
) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout,
    )
    if result.returncode != 0:
        raise AssertionError(
            "COMMANDE_ECHOUEE\n"
            + " ".join(command[:8])
            + "\n"
            + (result.stdout + "\n" + result.stderr)[-4000:]
        )
    return result


def package_versions() -> dict[str, str]:
    result = {}
    for package in ("pyarrow", "pandas", "ftfy", "regex"):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result[package] = "UNKNOWN"
    return result


assert Path("/content/persistent_storage").exists(), "Monter Google Drive."
assert FT_ROOT.is_dir(), FT_ROOT
K3_REPORT_ROOT.mkdir(parents=True, exist_ok=True)
K3_MODEL_ROOT.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# 3. Validation stricte de 20.K1 et 20.K2
# ---------------------------------------------------------------------------

def validate_k1() -> tuple[dict[str, Any], dict[str, Any]]:
    latest = read_small_json(K1_REPORT_ROOT / "LATEST_PASS.json")
    assert latest.get("cell") == "20.K1"
    assert latest.get("status") == "PASS_READY_FOR_20_K2"
    assert latest.get("run_id") == EXPECTED_K1_RUN_ID
    assert latest.get("report_sha256") == EXPECTED_K1_REPORT_SHA256
    assert latest.get("audio_downloaded") is False
    assert latest.get("audio_downloaded_bytes") == 0
    assert latest.get("dataset_rows_read") == 0
    assert latest.get("dataset_row_payload_downloaded_bytes") == 0

    run_dir = require_child(K1_REPORT_ROOT / EXPECTED_K1_RUN_ID, K1_REPORT_ROOT)
    report_path = require_child(latest["report"], run_dir)
    complete_path = require_child(latest["complete"], run_dir)
    assert report_path == (
        run_dir / "kenlm_text_source_inventory_20_K1.json"
    ).resolve()
    assert complete_path == (run_dir / "_COMPLETE.json").resolve()
    assert sha256_file(report_path) == EXPECTED_K1_REPORT_SHA256
    assert sha256_file(complete_path) == latest["complete_sha256"]

    report = read_small_json(report_path)
    complete = read_small_json(complete_path)
    for payload in (report, complete):
        assert payload.get("cell") == "20.K1"
        assert payload.get("status") == "PASS_READY_FOR_20_K2"
        assert payload.get("run_id") == EXPECTED_K1_RUN_ID
        assert payload.get("contract_sha256") == latest["contract_sha256"]

    # Le rapport principal de 20.K1 ne duplique pas necessairement ces champs.
    # Les preuves canoniques "metadata only / zero audio" sont figees dans
    # LATEST_PASS et _COMPLETE, exactement comme dans la validation 20.K2.
    assert complete.get("metadata_only") is True
    assert complete.get("audio_downloaded_bytes") == 0
    assert complete.get("dataset_rows_read") == 0
    assert complete.get("dataset_row_payload_downloaded_bytes") == 0
    assert sha256_bytes(
        canonical_json(report["contract"]).encode("utf-8")
    ) == report["contract_sha256"]

    expected_artifacts = {
        "kenlm_text_source_inventory_20_K1.json",
        "hf_source_inventory_20_K1.csv",
        "language_coverage_20_K1.csv",
        "kaggle_file_inventory_20_K1.csv",
    }
    assert set(complete["artifacts"]) == expected_artifacts
    for name, metadata in complete["artifacts"].items():
        artifact = require_child(run_dir / name, run_dir)
        assert artifact.stat().st_size == metadata["bytes"]
        assert sha256_file(artifact) == metadata["sha256"]

    return latest, report


def validate_k2() -> tuple[
    dict[str, Any],
    dict[str, Any],
    dict[str, Any],
    dict[str, Any],
    Path,
]:
    latest = read_small_json(K2_REPORT_ROOT / "LATEST_PASS.json")
    assert latest.get("cell") == "20.K2"
    assert latest.get("status") == "PASS_READY_FOR_20_K3_KENLM_BUILD"
    assert latest.get("run_id") == EXPECTED_K2_RUN_ID
    assert latest.get("contract_sha256") == EXPECTED_K2_CONTRACT_SHA256
    assert latest.get("audio_bytes_read") == 0
    corpus_root = require_child(latest["corpus_root"], PROJECT_ROOT)
    complete_path = require_child(latest["complete"], corpus_root)
    assert complete_path == (corpus_root / "_COMPLETE.json").resolve()
    assert sha256_file(complete_path) == latest["complete_sha256"]
    complete = read_small_json(complete_path)
    assert complete.get("cell") == "20.K2"
    assert complete.get("run_id") == EXPECTED_K2_RUN_ID
    assert complete.get("contract_sha256") == EXPECTED_K2_CONTRACT_SHA256
    assert complete.get("status") == "PASS_READY_FOR_20_K3_KENLM_BUILD"
    assert complete.get("audio_bytes_read") == 0
    for relative, metadata in complete["artifacts"].items():
        path = require_child(corpus_root / relative, corpus_root)
        assert path.is_file() and path.stat().st_size == metadata["bytes"]
        assert sha256_file(path) == metadata["sha256"]

    contract_path = require_child(corpus_root / "contract.json", corpus_root)
    report_path = require_child(latest["report"], K2_REPORT_ROOT)
    assert sha256_file(report_path) == latest["report_sha256"]
    contract = read_small_json(contract_path)
    report = read_small_json(report_path)
    assert sha256_bytes(canonical_json(contract).encode("utf-8")) == (
        EXPECTED_K2_CONTRACT_SHA256
    )
    assert report.get("cell") == "20.K2"
    assert report.get("status") == "PASS_READY_FOR_20_K3_KENLM_BUILD"
    assert report.get("run_id") == EXPECTED_K2_RUN_ID
    assert report.get("audio_bytes_read") == 0
    assert report["normalizer"]["version"] == NORMALIZER_VERSION
    return latest, complete, contract, report, corpus_root


K1_LATEST, K1_REPORT = validate_k1()
K2_LATEST, K2_COMPLETE, K2_CONTRACT, K2_REPORT, K2_CORPUS_ROOT = validate_k2()
print(
    "20.K2 verifie :",
    EXPECTED_K2_RUN_ID,
    EXPECTED_K2_CONTRACT_SHA256[:16],
)


# ---------------------------------------------------------------------------
# 4. Normaliseur exactement identique a 20.K2
# ---------------------------------------------------------------------------

MOJIBAKE_RE = re.compile(
    r"(?:Ã.|Â.|Ä.|Å.|â[€‚„…†‡ˆ‰Š‹ŒŽ‘’“”•–—˜™š›œžŸ]|ï¿½)"
)
APOSTROPHE_DASH_TRANSLATION = str.maketrans(
    {
        "’": "'",
        "‘": "'",
        "ʼ": "'",
        "`": "'",
        "–": " ",
        "—": " ",
        "‐": " ",
        "‑": " ",
    }
)


def suspect_score(text: str) -> int:
    return int(ftfy_badness(text)) + 4 * text.count("\ufffd") + len(
        MOJIBAKE_RE.findall(text)
    )


def letter_ratio(text: str) -> float:
    letters_marks = sum(
        char.isalpha() or unicodedata.category(char).startswith("M")
        for char in text
    )
    return letters_marks / max(1, len(text))


def repair_text(raw: Any) -> tuple[str | None, str]:
    if raw is None:
        return None, "reject_null"
    text = unicodedata.normalize("NFC", str(raw))
    if "\ufffd" in text:
        return None, "reject_replacement_char"
    if not MOJIBAKE_RE.search(text):
        return text, "unchanged"
    candidate = unicodedata.normalize(
        "NFC", ftfy.fix_text(text, normalization="NFC")
    )
    length_ratio = len(candidate) / max(1, len(text))
    safe = (
        "\ufffd" not in candidate
        and suspect_score(candidate) < suspect_score(text)
        and letter_ratio(candidate) + 0.02 >= letter_ratio(text)
        and 0.8 <= length_ratio <= 1.2
    )
    return (candidate, "fixed") if safe else (text, "suspect_unfixed")


def normalize_lm_text(raw: Any) -> tuple[str | None, str, list[str]]:
    repaired, mojibake_status = repair_text(raw)
    if repaired is None:
        return None, mojibake_status, []
    text = unicodedata.normalize(
        "NFC", repaired.translate(APOSTROPHE_DASH_TRANSLATION).casefold()
    )
    text = uregex.sub(r"[^\p{L}\p{M}\p{N}'-]+", " ", text)
    text = uregex.sub(
        r"(?<![\p{L}\p{M}])['-]|['-](?![\p{L}\p{M}])", " ", text
    )
    text = unicodedata.normalize("NFC", " ".join(text.split()))
    if not text:
        return None, "reject_empty", []
    flags = []
    if any(char.isdigit() for char in text):
        flags.append("contains_digit")
    if mojibake_status == "suspect_unfixed":
        flags.append("suspect_mojibake")
    if letter_ratio(text) < 0.45:
        return None, "reject_low_letter_ratio", flags
    return text, mojibake_status, flags


def chunk_normalized_text(text: str) -> list[str]:
    words = text.split()
    if len(words) <= MAX_WORDS_PER_LINE:
        return [text]
    chunks = [
        words[index : index + MAX_WORDS_PER_LINE]
        for index in range(0, len(words), MAX_WORDS_PER_LINE)
    ]
    if len(chunks) > 1 and len(chunks[-1]) < 8:
        needed = min(8 - len(chunks[-1]), len(chunks[-2]) - 1)
        if needed > 0:
            chunks[-1] = chunks[-2][-needed:] + chunks[-1]
            chunks[-2] = chunks[-2][:-needed]
    output = [" ".join(chunk) for chunk in chunks if chunk]
    assert output and all(
        1 <= len(chunk.split()) <= MAX_WORDS_PER_LINE for chunk in output
    )
    return output


def canonical_language(value: Any) -> str | None:
    normalized = str(value).strip().lower().replace("-", "_")
    base = normalized.split("_", 1)[0]
    for language, aliases in LANGUAGE_ALIASES.items():
        normalized_aliases = {alias.replace("-", "_") for alias in aliases}
        if (
            normalized == language
            or normalized in normalized_aliases
            or base == language
            or base in normalized_aliases
        ):
            return language
    return None


def normalize_split(value: Any) -> str:
    return str(value).strip().lower().replace("-", "_")


def exact_digest(language: str, text: str) -> bytes:
    return hashlib.sha256(f"{language}\0{text}".encode("utf-8")).digest()


def exact_set_sha256(values: Iterable[bytes]) -> str:
    digest = hashlib.sha256()
    for value in sorted(values):
        digest.update(value)
    return digest.hexdigest()


def is_text_type(data_type: pa.DataType) -> bool:
    if pa.types.is_dictionary(data_type):
        data_type = data_type.value_type
    return pa.types.is_string(data_type) or pa.types.is_large_string(data_type)


def is_audio_column(name: str) -> bool:
    lowered = name.lower()
    return any(pattern in lowered for pattern in AUDIO_COLUMN_PATTERNS)


def choose_text_column(schema: pa.Schema) -> str:
    lower_to_original = {name.lower(): name for name in schema.names}
    assert len(lower_to_original) == len(schema.names), schema.names
    for candidate in TEXT_PRIORITY:
        original = lower_to_original.get(candidate.lower())
        if original is not None:
            assert not is_audio_column(original)
            assert is_text_type(schema.field(original).type)
            return original
    raise AssertionError(f"Aucune colonne texte reconnue : {schema.names}")


def choose_optional_column(
    schema: pa.Schema, candidates: Iterable[str]
) -> str | None:
    lower_to_original = {name.lower(): name for name in schema.names}
    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]
    return None


def parquet_projection_sha256(path: Path, columns: list[str]) -> tuple[str, int]:
    assert columns and not any(is_audio_column(name) for name in columns)
    parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
    digest = hashlib.sha256()
    digest.update(canonical_json(columns).encode("utf-8"))
    rows = 0
    for batch in parquet_file.iter_batches(
        batch_size=8192, columns=columns, use_threads=False
    ):
        values = batch.to_pydict()
        for index in range(batch.num_rows):
            for column in columns:
                value = values[column][index]
                if value is None:
                    digest.update(b"\x00")
                else:
                    encoded = unicodedata.normalize("NFC", str(value)).encode(
                        "utf-8"
                    )
                    digest.update(b"\x01")
                    digest.update(len(encoded).to_bytes(8, "little"))
                    digest.update(encoded)
            rows += 1
    assert rows == parquet_file.metadata.num_rows
    return digest.hexdigest(), rows


# ---------------------------------------------------------------------------
# 5. Reconstruction locale exacte du held-out de 20.K2
# ---------------------------------------------------------------------------

def locate_v4_manifest() -> dict[str, Any]:
    records = [
        record
        for record in K1_REPORT["local_inventory"]
        if record.get("name") == "selected_records_v4"
    ]
    assert len(records) == 1
    record = records[0]
    path = require_child(record["path"], FT_ROOT)
    assert path.is_file() and path.suffix.lower() == ".parquet"
    digest = sha256_file(path)
    if record.get("sha256"):
        assert digest == record["sha256"]
    assert digest == K2_CONTRACT["v4_manifest_sha256"]
    parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
    schema = parquet_file.schema_arrow
    raw_text_column = choose_optional_column(schema, ("text", "text_raw"))
    text_norm_column = choose_optional_column(schema, ("text_norm",))
    text_column = choose_text_column(schema)
    language_column = choose_optional_column(schema, ("language", "lang"))
    split_column = choose_optional_column(schema, ("split",))
    assert language_column and split_column
    return {
        "path": str(path),
        "sha256": digest,
        "rows": int(parquet_file.metadata.num_rows),
        "text_column": text_norm_column or text_column,
        "raw_text_column": raw_text_column,
        "text_norm_column": text_norm_column,
        "language_column": language_column,
        "split_column": split_column,
    }


def reconstruct_heldout_plan(language: str) -> list[dict[str, Any]]:
    source = K2_CONTRACT["local_sources"][language]
    cache_root = require_child(source["cache_root"], PROJECT_ROOT / "dataset_caches")
    assert sha256_file(cache_root / "_COMPLETE.json") == source[
        "cache_complete_sha256"
    ]
    files: list[Path] = []
    version_root = cache_root / "version=0"
    if version_root.is_dir():
        files.extend(
            path
            for path in version_root.rglob("*.parquet")
            if "split=dev" in path.relative_to(version_root).parts
        )
    eval_long_root = cache_root / "eval_long"
    if eval_long_root.is_dir():
        files.extend(eval_long_root.rglob("*.parquet"))
    files = sorted({Path(path) for path in files})
    assert files, (language, "aucun held-out local")

    plan = []
    inventory = []
    for path in files:
        path = require_child(path, cache_root)
        assert path.is_file() and not path.is_symlink() and path.stat().st_size > 0
        parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
        schema = parquet_file.schema_arrow
        text_column = choose_text_column(schema)
        id_column = choose_optional_column(
            schema, ("sample_id", "source_id", "original_id", "id")
        )
        requested = [text_column]
        if id_column and id_column not in requested:
            requested.append(id_column)
        assert not any(is_audio_column(name) for name in requested)
        heldout_role = (
            "eval_long"
            if "eval_long" in path.relative_to(cache_root).parts
            else "dev"
        )
        projection_sha256, projection_rows = parquet_projection_sha256(
            path, requested
        )
        record = {
            "path": str(path),
            "relative_path": str(path.relative_to(cache_root)),
            "size_bytes": path.stat().st_size,
            "text_projection_sha256": projection_sha256,
            "rows_from_footer": int(parquet_file.metadata.num_rows),
            "text_column": text_column,
            "id_column": id_column,
            "requested_columns": requested,
            "heldout_role": heldout_role,
            "audio_columns_requested": [],
        }
        assert projection_rows == record["rows_from_footer"]
        plan.append(record)
        inventory.append(
            {
                key: record[key]
                for key in (
                    "relative_path",
                    "size_bytes",
                    "text_projection_sha256",
                    "rows_from_footer",
                    "requested_columns",
                    "heldout_role",
                )
            }
        )
    assert sum(record["rows_from_footer"] for record in plan) == source[
        "heldout_rows_from_footers"
    ]
    assert sha256_bytes(canonical_json(inventory).encode("utf-8")) == source[
        "heldout_inventory_sha256"
    ]
    return plan


def v4_group(split: str) -> str:
    if split in {"dev", "valid", "validation"}:
        return "v4_dev"
    if split == "local_test":
        return "v4_local_test"
    return "v4_other_heldout"


def build_heldout_local() -> tuple[dict[str, Any], dict[str, dict[str, Path]], Path]:
    local_root = Path("/content") / f"kenlm_v6_20_K3_heldout_{EXPECTED_K2_RUN_ID}"
    if local_root.exists():
        shutil.rmtree(local_root)
    local_root.mkdir(parents=True, exist_ok=True)

    entries: dict[str, dict[bytes, tuple[str, str]]] = {
        language: {} for language in LANGUAGES
    }
    stats = {language: Counter() for language in LANGUAGES}

    def add(language: str, raw: Any, group: str) -> None:
        normalized, status, _ = normalize_lm_text(raw)
        stats[language][f"normalizer|{status}"] += 1
        if normalized is None:
            stats[language]["rejected"] += 1
            return
        for text in chunk_normalized_text(normalized):
            digest = exact_digest(language, text)
            if digest in entries[language]:
                stats[language]["duplicate_exact"] += 1
                continue
            entries[language][digest] = (text, group)
            stats[language][f"group|{group}"] += 1

    v4 = locate_v4_manifest()
    parquet_file = pq.ParquetFile(v4["path"], memory_map=False, pre_buffer=False)
    columns = [v4["text_column"], v4["language_column"], v4["split_column"]]
    seen = 0
    for batch in parquet_file.iter_batches(
        batch_size=8192, columns=columns, use_threads=False
    ):
        values = batch.to_pydict()
        seen += batch.num_rows
        for index in range(batch.num_rows):
            split = normalize_split(values[v4["split_column"]][index])
            if split == "train":
                continue
            language = canonical_language(values[v4["language_column"]][index])
            assert language in LANGUAGES
            add(
                language,
                values[v4["text_column"]][index],
                v4_group(split),
            )
    assert seen == v4["rows"]

    plans = {}
    for language in LANGUAGES:
        plans[language] = reconstruct_heldout_plan(language)
        cache_root = Path(K2_CONTRACT["local_sources"][language]["cache_root"])
        for record in plans[language]:
            path = require_child(record["path"], cache_root)
            parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
            rows = 0
            for batch in parquet_file.iter_batches(
                batch_size=8192,
                columns=record["requested_columns"],
                use_threads=False,
            ):
                values = batch.to_pydict()
                rows += batch.num_rows
                for raw in values[record["text_column"]]:
                    add(language, raw, f"v5_{record['heldout_role']}")
            assert rows == record["rows_from_footer"]

    heldout_manifest: dict[str, Any] = {
        "schema": 1,
        "normalizer_version": NORMALIZER_VERSION,
        "split_policy": {
            "method": "sha256_digest_modulo",
            "modulus": AUDIT_MODULUS,
            "audit_residue": AUDIT_RESIDUE,
            "tune_fraction_nominal": (AUDIT_MODULUS - 1) / AUDIT_MODULUS,
            "audit_fraction_nominal": 1 / AUDIT_MODULUS,
        },
        "raw_text_persisted_to_drive": False,
        "languages": {},
    }
    local_files: dict[str, dict[str, Path]] = {}

    for language in LANGUAGES:
        expected = K2_REPORT["heldout_denylists"][language]
        values = entries[language]
        assert len(values) == expected["exact_unique"], (
            language,
            len(values),
            expected["exact_unique"],
        )
        assert exact_set_sha256(values) == expected["exact_set_sha256"]
        group_counts = Counter(group for _, group in values.values())
        assert sum(group_counts.values()) == len(values)

        partitions: dict[str, list[tuple[bytes, str]]] = {
            "combined": [],
            "tune": [],
            "audit": [],
        }
        for digest, (text, group) in sorted(values.items()):
            partitions["combined"].append((digest, text))
            residue = int.from_bytes(digest[:8], "big") % AUDIT_MODULUS
            split_name = "audit" if residue == AUDIT_RESIDUE else "tune"
            partitions[split_name].append((digest, text))
            partitions.setdefault(f"group__{group}", []).append((digest, text))
        assert len(partitions["tune"]) >= 50
        assert len(partitions["audit"]) >= 50
        assert len(partitions["tune"]) + len(partitions["audit"]) == len(values)

        language_dir = local_root / language
        language_dir.mkdir(parents=True, exist_ok=True)
        local_files[language] = {}
        file_meta = {}
        for name, rows in sorted(partitions.items()):
            path = language_dir / f"{name}.txt"
            line_digest = hashlib.sha256()
            word_count = 0
            with path.open("w", encoding="utf-8", newline="\n") as handle:
                for digest, text in rows:
                    handle.write(text + "\n")
                    line_digest.update(digest)
                    word_count += len(text.split())
            assert path.stat().st_size > 0
            local_files[language][name] = path
            file_meta[name] = {
                "lines": len(rows),
                "words": word_count,
                "exact_digest_sequence_sha256": line_digest.hexdigest(),
                "text_file_sha256": sha256_file(path),
            }
        heldout_manifest["languages"][language] = {
            "exact_unique": len(values),
            "exact_set_sha256": exact_set_sha256(values),
            "groups_first_observation_wins": dict(sorted(group_counts.items())),
            "partitions": file_meta,
            "stats": dict(sorted(stats[language].items())),
        }
        print(
            "Held-out",
            language,
            "| tune=",
            file_meta["tune"]["lines"],
            "| audit=",
            file_meta["audit"]["lines"],
            "| SHA=",
            expected["exact_set_sha256"][:12],
        )
    heldout_manifest["manifest_sha256"] = sha256_bytes(
        canonical_json(heldout_manifest).encode("utf-8")
    )
    return heldout_manifest, local_files, local_root


HELDOUT_MANIFEST, HELDOUT_FILES, HELDOUT_LOCAL_ROOT = build_heldout_local()


# ---------------------------------------------------------------------------
# 6. Contrat semantique 20.K3 et reprise atomique
# ---------------------------------------------------------------------------

K2_LANGUAGE_SUMMARY = {
    row["language"]: row for row in K2_REPORT["language_summary"]
}
assert set(K2_LANGUAGE_SUMMARY) == set(LANGUAGES)

CORPUS_INPUTS = {}
for language in LANGUAGES:
    relative = f"final/train_{language}.txt.gz"
    metadata = K2_COMPLETE["artifacts"][relative]
    CORPUS_INPUTS[language] = {
        "relative_path": relative,
        "sha256": metadata["sha256"],
        "bytes": metadata["bytes"],
        "rows": K2_LANGUAGE_SUMMARY[language]["final_rows"],
        "words": K2_LANGUAGE_SUMMARY[language]["final_words"],
    }

K3_CONTRACT = {
    "schema": 1,
    "cell": "20.K3",
    "pipeline_revision": PIPELINE_REVISION,
    "k1_run_id": EXPECTED_K1_RUN_ID,
    "k1_report_sha256": EXPECTED_K1_REPORT_SHA256,
    "k2_run_id": EXPECTED_K2_RUN_ID,
    "k2_contract_sha256": EXPECTED_K2_CONTRACT_SHA256,
    "k2_complete_sha256": K2_LATEST["complete_sha256"],
    "corpus_inputs": CORPUS_INPUTS,
    "heldout_manifest_sha256": HELDOUT_MANIFEST["manifest_sha256"],
    "heldout_exact_sets": {
        language: HELDOUT_MANIFEST["languages"][language]["exact_set_sha256"]
        for language in LANGUAGES
    },
    "normalizer": K2_REPORT["normalizer"],
    "kenlm": {
        "repository": KENLM_REPOSITORY,
        "commit": KENLM_COMMIT,
        "estimator": "modified_kneser_ney",
        "memory": LMPLZ_MEMORY,
        "binary_format": "trie_unquantized",
        "discount_fallback": "retry_only_for_discount_estimation_failure",
    },
    "candidate_grid": list(CANDIDATE_GRID),
    "unigrams": {
        "policy": "full_training_vocabulary",
        "sort": "descending_count_then_unicode_token",
    },
    "selection": {
        "max_v6_finalists_per_language": 2,
        "primary": "minimum_tune_cross_entropy_with_audit_guard",
        "compact_tolerance_ppl_ratio": PPL_TOLERANCE_RATIO,
        "compact_tolerance_bits": PPL_TOLERANCE_BITS,
        "audit_tolerance_bits": AUDIT_TOLERANCE_BITS,
        "historical_models_remain_eligible_for_20_K4": True,
    },
    "resource_gates": {
        "binary_max_bytes": MAX_BINARY_BYTES,
        "cold_query_rss_max_bytes": MAX_QUERY_RSS_BYTES,
        "final_edge_rss_bytes": FINAL_EDGE_RSS_BYTES,
        "final_edge_safety_margin": FINAL_EDGE_SAFETY_MARGIN,
        "k3_gate_is_lm_only": True,
    },
    "mutation_policy": {
        "historical_models_modified": False,
        "active_runtime_modified": False,
        "submission_created": False,
    },
}
K3_CONTRACT_SHA256 = sha256_bytes(canonical_json(K3_CONTRACT).encode("utf-8"))
K3_RUN_ID = K3_CONTRACT_SHA256[:16]
FINAL_RUN_DIR = K3_MODEL_ROOT / K3_RUN_ID
STAGING_RUN_DIR = K3_MODEL_ROOT / f"{K3_RUN_ID}.staging"
LOCAL_WORK_DIR = Path("/content") / f"kenlm_v6_20_K3_{K3_RUN_ID}"


def validate_completed_run(path: Path) -> dict[str, Any] | None:
    try:
        complete = read_small_json(path / "_COMPLETE.json")
        assert complete["cell"] == "20.K3"
        assert complete["run_id"] == K3_RUN_ID
        assert complete["contract_sha256"] == K3_CONTRACT_SHA256
        assert complete["status"] == "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING"
        assert read_small_json(path / "contract.json") == K3_CONTRACT
        heldout_manifest = read_small_json(path / "heldout_manifest.json")
        heldout_manifest_sha256 = heldout_manifest.pop("manifest_sha256")
        assert sha256_bytes(
            canonical_json(heldout_manifest).encode("utf-8")
        ) == heldout_manifest_sha256
        assert heldout_manifest_sha256 == K3_CONTRACT["heldout_manifest_sha256"]
        selection = read_small_json(path / "selection.json")
        assert set(selection) == set(LANGUAGES)
        required_artifacts = {
            "contract.json",
            "heldout_manifest.json",
            "toolchain.json",
            "candidate_metrics.csv",
            "historical_metrics.csv",
            "selection.json",
            "kenlm_build_eval_20_K3.json",
            *{
                f"manifests/{language}_{spec['name']}.json"
                for language in LANGUAGES
                for spec in CANDIDATE_GRID
            },
            *{
                f"shortlist/{language}/unigrams.txt"
                for language in LANGUAGES
            },
        }
        for language in LANGUAGES:
            finalists = selection[language]["v6_finalists"]
            assert 1 <= len(finalists) <= 2
            assert len({item["candidate"] for item in finalists}) == len(finalists)
            for item in finalists:
                expected_binary = (
                    f"shortlist/{language}/{item['candidate']}.binary"
                )
                assert item["binary_relative_path"] == expected_binary
                assert item["unigrams_relative_path"] == (
                    f"shortlist/{language}/unigrams.txt"
                )
                required_artifacts.add(expected_binary)
            for spec in CANDIDATE_GRID:
                manifest = read_small_json(
                    path / "manifests" / f"{language}_{spec['name']}.json"
                )
                retained = manifest["retention"]
                selected = spec["name"] in {
                    item["candidate"] for item in finalists
                }
                assert retained["selected_for_20_K4"] is selected
                if selected:
                    assert retained["retained_relative_path"] == (
                        f"shortlist/{language}/{spec['name']}.binary"
                    )
                else:
                    assert retained["retained_relative_path"] is None
        assert set(complete["artifacts"]) == required_artifacts
        assert not any(
            relative.endswith(".txt") and not relative.endswith("unigrams.txt")
            for relative in complete["artifacts"]
        )
        for relative, metadata in complete["artifacts"].items():
            artifact = require_child(path / relative, path)
            assert artifact.is_file() and artifact.stat().st_size == metadata["bytes"]
            assert sha256_file(artifact) == metadata["sha256"]
        return complete
    except Exception:
        return None


def repair_report_pointers() -> None:
    complete = validate_completed_run(FINAL_RUN_DIR)
    assert complete is not None
    report_run_dir = K3_REPORT_ROOT / K3_RUN_ID
    report_run_dir.mkdir(parents=True, exist_ok=True)
    for filename in (
        "kenlm_build_eval_20_K3.json",
        "candidate_metrics.csv",
        "historical_metrics.csv",
        "selection.json",
        "_COMPLETE.json",
    ):
        publish_file(FINAL_RUN_DIR / filename, report_run_dir / filename)
    latest = {
        "schema": 1,
        "cell": "20.K3",
        "status": "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING",
        "run_id": K3_RUN_ID,
        "contract_sha256": K3_CONTRACT_SHA256,
        "report": str(report_run_dir / "kenlm_build_eval_20_K3.json"),
        "report_sha256": sha256_file(
            report_run_dir / "kenlm_build_eval_20_K3.json"
        ),
        "complete": str(FINAL_RUN_DIR / "_COMPLETE.json"),
        "complete_sha256": sha256_file(FINAL_RUN_DIR / "_COMPLETE.json"),
        "model_root": str(FINAL_RUN_DIR),
        "historical_models_modified": False,
        "active_runtime_modified": False,
    }
    atomic_json(latest, K3_REPORT_ROOT / "LATEST_ATTEMPT.json")
    atomic_json(latest, K3_REPORT_ROOT / "LATEST.json")
    atomic_json(latest, K3_REPORT_ROOT / "LATEST_PASS.json")


# ---------------------------------------------------------------------------
# 7. Compilation reproductible de KenLM
# ---------------------------------------------------------------------------

def ensure_system_dependencies() -> None:
    packages = [
        "git",
        "build-essential",
        "cmake",
        "libboost-system-dev",
        "libboost-thread-dev",
        "libboost-program-options-dev",
        "libboost-test-dev",
        "libboost-iostreams-dev",
        "libeigen3-dev",
        "zlib1g-dev",
        "libbz2-dev",
        "liblzma-dev",
        "time",
    ]
    required_executables = ["git", "cmake", "g++", "/usr/bin/time"]
    required_headers = [
        Path("/usr/include/boost/program_options.hpp"),
        Path("/usr/include/boost/iostreams/filtering_stream.hpp"),
        Path("/usr/include/eigen3/Eigen/Core"),
        Path("/usr/include/zlib.h"),
        Path("/usr/include/bzlib.h"),
        Path("/usr/include/lzma.h"),
    ]
    executables_ready = all(
        shutil.which(item) or Path(item).exists()
        for item in required_executables
    )
    headers_ready = all(path.is_file() for path in required_headers)
    package_states = subprocess.run(
        [
            "dpkg-query",
            "-W",
            "-f=${binary:Package}\t${db:Status-Status}\n",
            *packages,
        ],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    installed_packages = set()
    if package_states.returncode == 0:
        for line in package_states.stdout.splitlines():
            name, _, status = line.partition("\t")
            if status.strip() == "installed":
                installed_packages.add(name.split(":", 1)[0])
    packages_ready = set(packages).issubset(installed_packages)
    if executables_ready and headers_ready and packages_ready:
        return
    print("Installation de la toolchain KenLM...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", *packages], check=True
    )


def ensure_kenlm() -> dict[str, Any]:
    ensure_system_dependencies()
    source = Path("/content/kenlm_20_K3_src")
    build = Path("/content/kenlm_20_K3_build")
    valid_source = False
    if (source / ".git").is_dir():
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=source,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
        )
        valid_source = result.returncode == 0 and result.stdout.strip() == KENLM_COMMIT
    if not valid_source:
        shutil.rmtree(source, ignore_errors=True)
        shutil.rmtree(build, ignore_errors=True)
        source.mkdir(parents=True, exist_ok=True)
        run_checked(["git", "init", "-q"], cwd=source)
        run_checked(["git", "remote", "add", "origin", KENLM_REPOSITORY], cwd=source)
        run_checked(
            ["git", "fetch", "-q", "--depth", "1", "origin", KENLM_COMMIT],
            cwd=source,
            timeout=600,
        )
        run_checked(["git", "checkout", "-q", "FETCH_HEAD"], cwd=source)
    assert run_checked(["git", "rev-parse", "HEAD"], cwd=source).stdout.strip() == (
        KENLM_COMMIT
    )

    executables = {
        "lmplz": build / "bin/lmplz",
        "build_binary": build / "bin/build_binary",
        "query": build / "bin/query",
    }
    if not all(path.is_file() and os.access(path, os.X_OK) for path in executables.values()):
        shutil.rmtree(build, ignore_errors=True)
        run_checked(
            [
                "cmake",
                "-S",
                str(source),
                "-B",
                str(build),
                "-DCMAKE_BUILD_TYPE=Release",
                "-DBUILD_TESTING=OFF",
                "-DKENLM_MAX_ORDER=6",
            ],
            timeout=600,
        )
        jobs = str(max(1, min(4, os.cpu_count() or 1)))
        run_checked(
            [
                "cmake",
                "--build",
                str(build),
                "--target",
                "lmplz",
                "build_binary",
                "query",
                "-j",
                jobs,
            ],
            timeout=1800,
        )
    assert all(path.is_file() and os.access(path, os.X_OK) for path in executables.values())
    compiler = run_checked(["g++", "--version"]).stdout.splitlines()[0]
    cmake = run_checked(["cmake", "--version"]).stdout.splitlines()[0]
    return {
        "repository": KENLM_REPOSITORY,
        "commit": KENLM_COMMIT,
        "source": str(source),
        "build": str(build),
        "executables": {
            name: {
                "path": str(path),
                "sha256": sha256_file(path),
                "bytes": path.stat().st_size,
            }
            for name, path in executables.items()
        },
        "compiler": compiler,
        "cmake": cmake,
        "python": sys.version.split()[0],
        "packages": package_versions(),
    }


# ---------------------------------------------------------------------------
# 8. Audit des corpus train et vocabulaire complet
# ---------------------------------------------------------------------------

def audit_train_and_build_unigrams(
    language: str,
    staging_dir: Path,
    local_dir: Path,
) -> dict[str, Any]:
    source_meta = CORPUS_INPUTS[language]
    source = require_child(K2_CORPUS_ROOT / source_meta["relative_path"], K2_CORPUS_ROOT)
    assert source.stat().st_size == source_meta["bytes"]
    assert sha256_file(source) == source_meta["sha256"]
    heldout_exact_sha = K2_REPORT["heldout_denylists"][language][
        "exact_set_sha256"
    ]
    assert heldout_exact_sha == HELDOUT_MANIFEST["languages"][language][
        "exact_set_sha256"
    ]
    heldout_values = set()
    # Les textes sont locaux seulement; ils servent ici a prouver zero overlap.
    with HELDOUT_FILES[language]["combined"].open("r", encoding="utf-8") as handle:
        for line in handle:
            text = line.rstrip("\n")
            heldout_values.add(exact_digest(language, text))
    assert exact_set_sha256(heldout_values) == heldout_exact_sha

    word_counter: Counter[str] = Counter()
    rows = 0
    words = 0
    overlaps = 0
    sequence = hashlib.sha256()
    corpus_dir = local_dir / "corpora"
    corpus_dir.mkdir(parents=True, exist_ok=True)
    local_train_text = corpus_dir / f"train_{language}.txt"
    temporary_train_text = local_train_text.with_name(
        local_train_text.name + f".tmp-{os.getpid()}"
    )
    if temporary_train_text.exists():
        temporary_train_text.unlink()
    with gzip.open(
        source, "rt", encoding="utf-8", errors="strict", newline=""
    ) as handle, temporary_train_text.open(
        "w", encoding="utf-8", newline="\n"
    ) as plain_handle:
        for line in handle:
            assert line.endswith("\n"), (language, "ligne sans newline")
            text = line[:-1]
            assert text and "\r" not in text and "\n" not in text
            normalized, _, _ = normalize_lm_text(text)
            assert normalized == text, (language, "normalisation non idempotente")
            digest = exact_digest(language, text)
            overlaps += int(digest in heldout_values)
            encoded = text.encode("utf-8")
            sequence.update(len(encoded).to_bytes(8, "little"))
            sequence.update(encoded)
            tokens = text.split()
            word_counter.update(tokens)
            plain_handle.write(text + "\n")
            rows += 1
            words += len(tokens)
    os.replace(temporary_train_text, local_train_text)
    assert rows == source_meta["rows"], (language, rows, source_meta["rows"])
    assert words == source_meta["words"], (language, words, source_meta["words"])
    assert overlaps == 0, (language, "TRAIN_HELDOUT_EXACT_OVERLAP", overlaps)
    assert word_counter

    local_unigrams = local_dir / f"{language}_unigrams.txt"
    local_unigrams.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(word_counter.items(), key=lambda item: (-item[1], item[0]))
    with local_unigrams.open("w", encoding="utf-8", newline="\n") as handle:
        for token, _ in ordered:
            assert token and "\n" not in token and "\r" not in token
            handle.write(token + "\n")
    drive_unigrams = staging_dir / "unigrams" / f"{language}.txt"
    publish_file(local_unigrams, drive_unigrams)
    return {
        "language": language,
        "rows": rows,
        "words": words,
        "unique_words": len(ordered),
        "train_sequence_sha256": sequence.hexdigest(),
        "local_plain_text": {
            "bytes": local_train_text.stat().st_size,
            "sha256": sha256_file(local_train_text),
            "rows": rows,
            "words": words,
            "persisted_to_drive": False,
        },
        "train_heldout_exact_overlaps": overlaps,
        "unigrams": {
            "relative_path": f"unigrams/{language}.txt",
            "bytes": drive_unigrams.stat().st_size,
            "sha256": sha256_file(drive_unigrams),
            "coverage_weighted": 1.0,
            "vocabulary_truncated": False,
        },
    }


# ---------------------------------------------------------------------------
# 9. Construction, query et RSS dans des processus propres
# ---------------------------------------------------------------------------

QUERY_PATTERNS = {
    "ppl_including_oov": re.compile(
        r"Perplexity including OOVs:\s*([^\s]+)", re.IGNORECASE
    ),
    "ppl_excluding_oov": re.compile(
        r"Perplexity excluding OOVs:\s*([^\s]+)", re.IGNORECASE
    ),
    "oovs": re.compile(r"^OOVs:\s*(\d+)\s*$", re.MULTILINE),
    "tokens": re.compile(r"^Tokens:\s*(\d+)\s*$", re.MULTILINE),
}
RSS_PATTERN = re.compile(r"Maximum resident set size \(kbytes\):\s*(\d+)")


def parse_query_summary(stdout: str) -> dict[str, Any]:
    values = {}
    for name, pattern in QUERY_PATTERNS.items():
        match = pattern.search(stdout)
        assert match, (name, stdout[-2000:])
        values[name] = match.group(1)
    ppl_including = float(values["ppl_including_oov"])
    ppl_excluding = float(values["ppl_excluding_oov"])
    oovs = int(values["oovs"])
    tokens = int(values["tokens"])
    assert math.isfinite(ppl_including) and ppl_including > 0
    assert 0 <= oovs <= tokens and tokens > 0
    excluding_valid = math.isfinite(ppl_excluding) and ppl_excluding > 0
    return {
        "ppl_including_oov": ppl_including,
        "ppl_excluding_oov": ppl_excluding if excluding_valid else None,
        "ppl_excluding_oov_valid": excluding_valid,
        "cross_entropy_bits": math.log2(ppl_including),
        "oovs": oovs,
        "tokens": tokens,
        "oov_rate_all_scored_tokens": oovs / tokens,
    }


def query_binary(
    query_executable: Path,
    binary: Path,
    text_path: Path,
    *,
    measure_rss: bool,
) -> dict[str, Any]:
    command = [
        str(query_executable),
        "-v",
        "summary",
        "-l",
        "lazy",
        str(binary),
    ]
    if measure_rss:
        command = ["/usr/bin/time", "-v", *command]
    env = os.environ.copy()
    env.update({"LC_ALL": "C", "LANG": "C", "OMP_NUM_THREADS": "1"})
    started = time.monotonic()
    with text_path.open("rb") as input_handle:
        result = subprocess.run(
            command,
            stdin=input_handle,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=env,
            timeout=1800,
        )
    elapsed = time.monotonic() - started
    stdout = result.stdout.decode("utf-8", errors="replace")
    stderr = result.stderr.decode("utf-8", errors="replace")
    assert result.returncode == 0, (result.returncode, stderr[-3000:])
    metrics = parse_query_summary(stdout)
    metrics["elapsed_seconds"] = elapsed
    metrics["stdout_contains_only_summary"] = (
        "=" not in stdout and "Total:" not in stdout
    )
    assert metrics["stdout_contains_only_summary"]
    if measure_rss:
        match = RSS_PATTERN.search(stderr)
        assert match, stderr[-4000:]
        metrics["max_rss_kib"] = int(match.group(1))
        metrics["max_rss_bytes"] = int(match.group(1)) * 1024
    return metrics


def evaluate_binary(
    language: str,
    binary: Path,
    query_executable: Path,
) -> dict[str, Any]:
    evaluations = {}
    for name in ("combined", "tune", "audit"):
        evaluations[name] = query_binary(
            query_executable,
            binary,
            HELDOUT_FILES[language][name],
            measure_rss=name == "combined",
        )
        partition = HELDOUT_MANIFEST["languages"][language]["partitions"][name]
        expected_words = partition["words"]
        expected_tokens = expected_words + partition["lines"]
        assert evaluations[name]["tokens"] == expected_tokens, (
            language,
            name,
            evaluations[name]["tokens"],
            expected_tokens,
        )
        evaluations[name]["lexical_tokens"] = expected_words
        evaluations[name]["sentence_end_tokens"] = partition["lines"]
        evaluations[name]["oov_rate"] = evaluations[name]["oovs"] / max(
            1, expected_words
        )
    groups = {}
    for name, path in sorted(HELDOUT_FILES[language].items()):
        if not name.startswith("group__"):
            continue
        role = name.split("__", 1)[1]
        groups[role] = query_binary(
            query_executable, binary, path, measure_rss=False
        )
        partition = HELDOUT_MANIFEST["languages"][language]["partitions"][name]
        expected_words = partition["words"]
        expected_tokens = expected_words + partition["lines"]
        assert groups[role]["tokens"] == expected_tokens
        groups[role]["lexical_tokens"] = expected_words
        groups[role]["sentence_end_tokens"] = partition["lines"]
        groups[role]["oov_rate"] = groups[role]["oovs"] / max(1, expected_words)
    return {
        "combined": evaluations["combined"],
        "tune": evaluations["tune"],
        "audit": evaluations["audit"],
        "groups": groups,
    }


def valid_candidate_checkpoint(
    language: str,
    spec: dict[str, Any],
    staging_dir: Path,
) -> dict[str, Any] | None:
    manifest_path = staging_dir / "manifests" / f"{language}_{spec['name']}.json"
    binary_path = staging_dir / "candidates" / language / f"{spec['name']}.binary"
    try:
        manifest = read_small_json(manifest_path)
        assert manifest["phase"] == "kenlm_candidate_complete"
        assert manifest["language"] == language
        assert manifest["candidate"] == spec
        assert manifest["contract_sha256"] == K3_CONTRACT_SHA256
        assert manifest["kenlm_commit"] == KENLM_COMMIT
        assert manifest["corpus_sha256"] == CORPUS_INPUTS[language]["sha256"]
        assert manifest["heldout_manifest_sha256"] == HELDOUT_MANIFEST[
            "manifest_sha256"
        ]
        assert binary_path.is_file()
        assert binary_path.stat().st_size == manifest["binary"]["bytes"]
        assert sha256_file(binary_path) == manifest["binary"]["sha256"]
        assert manifest["binary"]["working_relative_path"] == (
            f"candidates/{language}/{spec['name']}.binary"
        )
        for split in ("combined", "tune", "audit"):
            metrics = manifest["evaluation"][split]
            assert math.isfinite(metrics["cross_entropy_bits"])
            partition = HELDOUT_MANIFEST["languages"][language]["partitions"][
                split
            ]
            assert metrics["tokens"] == partition["words"] + partition["lines"]
            assert metrics["lexical_tokens"] == partition["words"]
        expected_groups = {
            name.split("__", 1)[1]
            for name in HELDOUT_FILES[language]
            if name.startswith("group__")
        }
        assert set(manifest["evaluation"]["groups"]) == expected_groups
        for role in expected_groups:
            metrics = manifest["evaluation"]["groups"][role]
            partition = HELDOUT_MANIFEST["languages"][language]["partitions"][
                f"group__{role}"
            ]
            assert metrics["tokens"] == partition["words"] + partition["lines"]
        expected_gate = {
            "binary_under_512_mib": manifest["binary"]["bytes"]
            <= MAX_BINARY_BYTES,
            "cold_query_rss_under_1_gib": manifest["evaluation"]["combined"][
                "max_rss_bytes"
            ]
            <= MAX_QUERY_RSS_BYTES,
        }
        assert manifest["resource_gate"]["binary_under_512_mib"] == expected_gate[
            "binary_under_512_mib"
        ]
        assert manifest["resource_gate"]["cold_query_rss_under_1_gib"] == (
            expected_gate["cold_query_rss_under_1_gib"]
        )
        assert manifest["resource_gate"]["pass"] == all(expected_gate.values())
        return manifest
    except Exception:
        return None


def discount_failure(stderr: str) -> bool:
    lowered = stderr.casefold()
    discount_markers = (
        "discount",
        "could not calculate",
        "unigram count",
        "bad discount",
    )
    resource_markers = (
        "out of memory",
        "no space left",
        "killed",
        "cannot allocate memory",
    )
    return any(marker in lowered for marker in discount_markers) and not any(
        marker in lowered for marker in resource_markers
    )


def run_with_heartbeat(
    command: list[str],
    *,
    stdin_handle: Any = None,
    stdout_handle: Any = None,
    env: dict[str, str] | None = None,
    timeout_seconds: int = 7200,
    label: str,
    log_path: Path,
) -> subprocess.CompletedProcess[bytes]:
    """Execute sans accumuler les logs et imprime un battement chaque minute."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    next_heartbeat = 60.0
    with log_path.open("wb") as log_handle:
        process = subprocess.Popen(
            command,
            stdin=stdin_handle,
            stdout=stdout_handle if stdout_handle is not None else log_handle,
            stderr=log_handle,
            env=env,
        )
        while process.poll() is None:
            elapsed = time.monotonic() - started
            if elapsed >= timeout_seconds:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise TimeoutError(f"{label} depasse {timeout_seconds / 60:.0f} min")
            if elapsed >= next_heartbeat:
                print(f"  {label} en cours | {elapsed / 60:.1f} min")
                next_heartbeat += 60.0
            time.sleep(5)
        returncode = process.returncode
    stderr = log_path.read_bytes()
    return subprocess.CompletedProcess(command, returncode, b"", stderr)


def build_candidate(
    language: str,
    spec: dict[str, Any],
    staging_dir: Path,
    local_dir: Path,
    toolchain: dict[str, Any],
    train_audit: dict[str, Any],
) -> dict[str, Any]:
    resumed = valid_candidate_checkpoint(language, spec, staging_dir)
    if resumed is not None and resumed.get("local_plain_text_sha256") != (
        train_audit["local_plain_text"]["sha256"]
    ):
        resumed = None
    if resumed is not None:
        print(
            f"[reprise] {language} {spec['name']} | "
            f"PPL={resumed['evaluation']['combined']['ppl_including_oov']:.3f}"
        )
        return resumed

    lmplz = Path(toolchain["executables"]["lmplz"]["path"])
    build_binary = Path(toolchain["executables"]["build_binary"]["path"])
    query = Path(toolchain["executables"]["query"]["path"])
    corpus = local_dir / "corpora" / f"train_{language}.txt"
    assert corpus.is_file() and corpus.stat().st_size == train_audit[
        "local_plain_text"
    ]["bytes"]
    assert sha256_file(corpus) == train_audit["local_plain_text"]["sha256"]
    candidate_dir = local_dir / "candidates" / language
    temp_dir = local_dir / "tmp" / language / spec["name"]
    candidate_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(parents=True, exist_ok=True)
    arpa = candidate_dir / f"{spec['name']}.arpa"
    local_binary = candidate_dir / f"{spec['name']}.binary"
    for path in (arpa, local_binary):
        if path.exists():
            path.unlink()

    base_command = [
        str(lmplz),
        "-o",
        str(spec["order"]),
        "-S",
        LMPLZ_MEMORY,
        "-T",
        str(temp_dir),
    ]
    if spec["prune"]:
        base_command.extend(["--prune", *[str(value) for value in spec["prune"]]])
    env = os.environ.copy()
    env.update({"LC_ALL": "C", "LANG": "C", "OMP_NUM_THREADS": "1"})
    first_failure = None
    fallback_used = False
    started = time.monotonic()

    def estimate(command: list[str]) -> subprocess.CompletedProcess[bytes]:
        if arpa.exists():
            arpa.unlink()
        shutil.rmtree(temp_dir, ignore_errors=True)
        temp_dir.mkdir(parents=True, exist_ok=True)
        with corpus.open("rb") as input_handle, arpa.open("wb") as output_handle:
            return run_with_heartbeat(
                command,
                stdin_handle=input_handle,
                stdout_handle=output_handle,
                env=env,
                timeout_seconds=7200,
                label=f"lmplz {language}|{spec['name']}",
                log_path=temp_dir / "lmplz.log",
            )

    try:
        result = estimate(base_command)
        if result.returncode != 0:
            stderr = result.stderr.decode("utf-8", errors="replace")
            first_failure = {
                "returncode": result.returncode,
                "stderr_sha256": sha256_bytes(result.stderr),
                "classified_as_discount_failure": discount_failure(stderr),
            }
            assert first_failure["classified_as_discount_failure"], stderr[-4000:]
            fallback_used = True
            result = estimate([*base_command, "--discount_fallback"])
        assert result.returncode == 0, result.stderr.decode(
            "utf-8", errors="replace"
        )[-4000:]
        assert arpa.is_file() and arpa.stat().st_size > 0

        binary_result = run_with_heartbeat(
            [str(build_binary), "trie", str(arpa), str(local_binary)],
            env=env,
            timeout_seconds=7200,
            label=f"build_binary {language}|{spec['name']}",
            log_path=temp_dir / "build_binary.log",
        )
        assert binary_result.returncode == 0, binary_result.stderr.decode(
            "utf-8", errors="replace"
        )[-4000:]
        assert local_binary.is_file() and local_binary.stat().st_size > 0
        binary_sha256 = sha256_file(local_binary)
        evaluation = evaluate_binary(language, local_binary, query)
        binary_bytes = local_binary.stat().st_size
        resource_gate = {
            "binary_under_512_mib": binary_bytes <= MAX_BINARY_BYTES,
            "cold_query_rss_under_1_gib": evaluation["combined"][
                "max_rss_bytes"
            ]
            <= MAX_QUERY_RSS_BYTES,
        }
        resource_gate["pass"] = all(resource_gate.values())
        drive_binary = (
            staging_dir / "candidates" / language / f"{spec['name']}.binary"
        )
        publish_file(local_binary, drive_binary)
        assert sha256_file(drive_binary) == binary_sha256
        manifest = {
            "schema": 1,
            "phase": "kenlm_candidate_complete",
            "language": language,
            "candidate": spec,
            "contract_sha256": K3_CONTRACT_SHA256,
            "kenlm_commit": KENLM_COMMIT,
            "corpus_sha256": CORPUS_INPUTS[language]["sha256"],
            "local_plain_text_sha256": train_audit["local_plain_text"]["sha256"],
            "heldout_manifest_sha256": HELDOUT_MANIFEST["manifest_sha256"],
            "command": {
                "order": spec["order"],
                "prune": spec["prune"],
                "memory": LMPLZ_MEMORY,
                "binary_format": "trie",
                "discount_fallback_used": fallback_used,
                "first_failure": first_failure,
            },
            "tool_sha256": {
                name: record["sha256"]
                for name, record in toolchain["executables"].items()
            },
            "binary": {
                "working_relative_path": (
                    f"candidates/{language}/{spec['name']}.binary"
                ),
                "bytes": binary_bytes,
                "sha256": binary_sha256,
            },
            "evaluation": evaluation,
            "resource_gate": resource_gate,
            "elapsed_build_and_eval_minutes": (time.monotonic() - started) / 60,
            "raw_corpus_text_in_manifest": False,
            "retention": {
                "selected_for_20_K4": False,
                "retained_relative_path": None,
                "unigrams_relative_path": None,
            },
        }
        atomic_json(
            manifest,
            staging_dir / "manifests" / f"{language}_{spec['name']}.json",
        )
        print(
            f"✅ {language} {spec['name']} | "
            f"PPL={evaluation['combined']['ppl_including_oov']:.3f} | "
            f"OOV={evaluation['combined']['oov_rate']:.2%} | "
            f"{binary_bytes / 2**20:.1f} Mio | "
            f"RSS={evaluation['combined']['max_rss_bytes'] / 2**30:.2f} Gio"
        )
        return manifest
    finally:
        if arpa.exists():
            arpa.unlink()
        if local_binary.exists():
            local_binary.unlink()
        shutil.rmtree(temp_dir, ignore_errors=True)
        gc.collect()


def evaluate_historical(
    language: str,
    local_dir: Path,
    toolchain: dict[str, Any],
) -> dict[str, Any]:
    record = HISTORICAL_MODELS[language]
    source = require_child(record["path"], FT_ROOT)
    assert source.is_file() and source.stat().st_size > 0
    digest = sha256_file(source)
    local_binary = local_dir / "historical" / language / source.name
    copy_verified(source, local_binary, digest)
    evaluation = evaluate_binary(
        language,
        local_binary,
        Path(toolchain["executables"]["query"]["path"]),
    )
    return {
        "language": language,
        "family": record["family"],
        "path": str(source),
        "sha256": digest,
        "bytes": source.stat().st_size,
        "evaluation": evaluation,
        "comparison_note": (
            "PPL informative seulement : le vocabulaire historique differe du V6"
        ),
    }


# ---------------------------------------------------------------------------
# 10. Selection V6, publication atomique et rapport
# ---------------------------------------------------------------------------

def candidate_rows(manifests: dict[str, dict[str, dict[str, Any]]]) -> list[dict[str, Any]]:
    rows = []
    for language in LANGUAGES:
        for name, manifest in manifests[language].items():
            evaluation = manifest["evaluation"]
            rows.append(
                {
                    "language": language,
                    "candidate": name,
                    "order": manifest["candidate"]["order"],
                    "prune": " ".join(map(str, manifest["candidate"]["prune"])),
                    "discount_fallback_used": manifest["command"][
                        "discount_fallback_used"
                    ],
                    "combined_ppl": evaluation["combined"]["ppl_including_oov"],
                    "combined_bits": evaluation["combined"]["cross_entropy_bits"],
                    "combined_oov_rate": evaluation["combined"]["oov_rate"],
                    "tune_ppl": evaluation["tune"]["ppl_including_oov"],
                    "tune_bits": evaluation["tune"]["cross_entropy_bits"],
                    "audit_ppl": evaluation["audit"]["ppl_including_oov"],
                    "audit_bits": evaluation["audit"]["cross_entropy_bits"],
                    "binary_mib": manifest["binary"]["bytes"] / 2**20,
                    "query_rss_gib": evaluation["combined"]["max_rss_bytes"] / 2**30,
                    "resource_gate_pass": manifest["resource_gate"]["pass"],
                }
            )
    return rows


def historical_rows(manifests: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    rows = []
    for language in LANGUAGES:
        manifest = manifests[language]
        evaluation = manifest["evaluation"]
        rows.append(
            {
                "language": language,
                "family": manifest["family"],
                "combined_ppl": evaluation["combined"]["ppl_including_oov"],
                "combined_bits": evaluation["combined"]["cross_entropy_bits"],
                "combined_oov_rate": evaluation["combined"]["oov_rate"],
                "tune_ppl": evaluation["tune"]["ppl_including_oov"],
                "audit_ppl": evaluation["audit"]["ppl_including_oov"],
                "binary_mib": manifest["bytes"] / 2**20,
                "query_rss_gib": evaluation["combined"]["max_rss_bytes"] / 2**30,
                "directly_comparable_to_v6": False,
            }
        )
    return rows


def select_shortlist(
    manifests: dict[str, dict[str, dict[str, Any]]]
) -> dict[str, Any]:
    selection = {}
    for language in LANGUAGES:
        candidates = [
            manifest
            for manifest in manifests[language].values()
            if manifest["resource_gate"]["pass"]
        ]
        assert candidates, (language, "aucun candidat V6 conforme")
        minimum_audit = min(
            manifest["evaluation"]["audit"]["cross_entropy_bits"]
            for manifest in candidates
        )
        guarded = [
            manifest
            for manifest in candidates
            if manifest["evaluation"]["audit"]["cross_entropy_bits"]
            <= minimum_audit + AUDIT_TOLERANCE_BITS
        ]
        assert guarded
        primary = min(
            guarded,
            key=lambda manifest: (
                manifest["evaluation"]["tune"]["cross_entropy_bits"],
                manifest["binary"]["bytes"],
                manifest["candidate"]["name"],
            ),
        )
        primary_bits = primary["evaluation"]["tune"]["cross_entropy_bits"]
        compact_pool = [
            manifest
            for manifest in guarded
            if manifest["evaluation"]["tune"]["cross_entropy_bits"]
            <= primary_bits + PPL_TOLERANCE_BITS
        ]
        compact = min(
            compact_pool,
            key=lambda manifest: (
                manifest["binary"]["bytes"],
                manifest["evaluation"]["tune"]["cross_entropy_bits"],
                manifest["candidate"]["name"],
            ),
        )
        chosen = [primary]
        if compact["candidate"]["name"] != primary["candidate"]["name"]:
            chosen.append(compact)
        selection[language] = {
            "historical_baseline": {
                "family": HISTORICAL_MODELS[language]["family"],
                "path": str(HISTORICAL_MODELS[language]["path"]),
                "remains_eligible_for_20_K4": True,
            },
            "minimum_v6_audit_bits": minimum_audit,
            "primary_rule": "minimum tune bits under audit guard",
            "compact_rule": (
                "smallest binary within 1% PPL of primary and under audit guard"
            ),
            "v6_finalists": [
                {
                    "candidate": manifest["candidate"]["name"],
                    "combined_ppl": manifest["evaluation"]["combined"][
                        "ppl_including_oov"
                    ],
                    "combined_bits": manifest["evaluation"]["combined"][
                        "cross_entropy_bits"
                    ],
                    "tune_ppl": manifest["evaluation"]["tune"][
                        "ppl_including_oov"
                    ],
                    "tune_bits": manifest["evaluation"]["tune"][
                        "cross_entropy_bits"
                    ],
                    "audit_ppl": manifest["evaluation"]["audit"][
                        "ppl_including_oov"
                    ],
                    "audit_bits": manifest["evaluation"]["audit"][
                        "cross_entropy_bits"
                    ],
                    "binary_bytes": manifest["binary"]["bytes"],
                    "binary_sha256": manifest["binary"]["sha256"],
                    "binary_relative_path": (
                        f"shortlist/{language}/"
                        f"{manifest['candidate']['name']}.binary"
                    ),
                    "unigrams_relative_path": (
                        f"shortlist/{language}/unigrams.txt"
                    ),
                }
                for manifest in chosen
            ],
        }
    return selection


def annotate_candidate_retention(
    manifests: dict[str, dict[str, dict[str, Any]]],
    selection: dict[str, Any],
    staging_dir: Path,
) -> None:
    for language in LANGUAGES:
        selected = {
            item["candidate"]: item
            for item in selection[language]["v6_finalists"]
        }
        for name, manifest in manifests[language].items():
            finalist = selected.get(name)
            manifest["retention"] = {
                "selected_for_20_K4": finalist is not None,
                "retained_relative_path": (
                    finalist["binary_relative_path"] if finalist else None
                ),
                "unigrams_relative_path": (
                    finalist["unigrams_relative_path"] if finalist else None
                ),
                "working_binary_removed_after_publication": True,
            }
            if finalist:
                retained = staging_dir / finalist["binary_relative_path"]
                assert retained.is_file()
                assert sha256_file(retained) == manifest["binary"]["sha256"]
            atomic_json(
                manifest,
                staging_dir / "manifests" / f"{language}_{name}.json",
            )


def cleanup_unlisted_work_files(path: Path) -> None:
    """Retire seulement les binaires transitoires exclus des artifacts finaux."""
    shutil.rmtree(path / "candidates", ignore_errors=True)
    shutil.rmtree(path / "unigrams", ignore_errors=True)


def main() -> None:
    print("Contrat 20.K3 :", K3_CONTRACT_SHA256)
    print("Run ID         :", K3_RUN_ID)
    if FINAL_RUN_DIR.exists():
        assert validate_completed_run(FINAL_RUN_DIR) is not None, (
            "FINAL_RUN_DIR_EXISTE_MAIS_EST_INVALIDE",
            FINAL_RUN_DIR,
        )
        cleanup_unlisted_work_files(FINAL_RUN_DIR)
        assert validate_completed_run(FINAL_RUN_DIR) is not None
        repair_report_pointers()
        report = read_small_json(FINAL_RUN_DIR / "kenlm_build_eval_20_K3.json")
        print("[reprise] 20.K3 deja complet et reverifie.")
        print("STATUT 20.K3 :", report["status"])
        print("Etape suivante : 20.K4 — tuning WER acoustique V6 vs historiques")
        return

    assert shutil.disk_usage("/content").free >= MIN_LOCAL_FREE_BYTES, (
        "Au moins 25 Gio libres sont requis sur /content"
    )
    assert shutil.disk_usage(K3_MODEL_ROOT).free >= MIN_DRIVE_FREE_BYTES, (
        "Au moins 16 Gio libres sont requis sur Drive pendant la construction"
    )
    if STAGING_RUN_DIR.exists():
        existing_contract = STAGING_RUN_DIR / "contract.json"
        if existing_contract.is_file():
            assert read_small_json(existing_contract) == K3_CONTRACT
    for directory in (
        STAGING_RUN_DIR,
        STAGING_RUN_DIR / "candidates",
        STAGING_RUN_DIR / "manifests",
        STAGING_RUN_DIR / "unigrams",
        LOCAL_WORK_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    atomic_json(K3_CONTRACT, STAGING_RUN_DIR / "contract.json")
    atomic_json(HELDOUT_MANIFEST, STAGING_RUN_DIR / "heldout_manifest.json")

    toolchain = ensure_kenlm()
    atomic_json(toolchain, STAGING_RUN_DIR / "toolchain.json")
    print("KenLM fige :", KENLM_COMMIT)

    train_audits = {}
    for language in LANGUAGES:
        print("Audit train/unigrammes :", language)
        train_audits[language] = audit_train_and_build_unigrams(
            language, STAGING_RUN_DIR, LOCAL_WORK_DIR
        )

    candidate_manifests: dict[str, dict[str, dict[str, Any]]] = {
        language: {} for language in LANGUAGES
    }
    total_candidates = len(LANGUAGES) * len(CANDIDATE_GRID)
    candidate_index = 0
    for language in LANGUAGES:
        for spec in CANDIDATE_GRID:
            candidate_index += 1
            print("\n" + "=" * 76)
            print(
                f"[{candidate_index:02d}/{total_candidates}] {language} | "
                f"{spec['name']} | ordre={spec['order']} | prune={spec['prune']}"
            )
            candidate_manifests[language][spec["name"]] = build_candidate(
                language,
                dict(spec),
                STAGING_RUN_DIR,
                LOCAL_WORK_DIR,
                toolchain,
                train_audits[language],
            )

    print("\nEvaluation informative des KenLM historiques...")
    historical_manifests = {
        language: evaluate_historical(language, LOCAL_WORK_DIR, toolchain)
        for language in LANGUAGES
    }
    selection = select_shortlist(candidate_manifests)

    # Copier d'abord les finalistes et leurs unigrammes; les candidats de
    # travail sont supprimes seulement apres verification de chaque copie.
    for language in LANGUAGES:
        source_unigrams = STAGING_RUN_DIR / "unigrams" / f"{language}.txt"
        target_unigrams = STAGING_RUN_DIR / "shortlist" / language / "unigrams.txt"
        publish_file(source_unigrams, target_unigrams)
        assert sha256_file(target_unigrams) == sha256_file(source_unigrams)
        for finalist in selection[language]["v6_finalists"]:
            name = finalist["candidate"]
            source_binary = (
                STAGING_RUN_DIR / "candidates" / language / f"{name}.binary"
            )
            target_binary = (
                STAGING_RUN_DIR / "shortlist" / language / f"{name}.binary"
            )
            publish_file(source_binary, target_binary)
            assert sha256_file(target_binary) == finalist["binary_sha256"]

    annotate_candidate_retention(
        candidate_manifests, selection, STAGING_RUN_DIR
    )

    candidate_frame = pd.DataFrame(candidate_rows(candidate_manifests)).sort_values(
        ["language", "combined_bits", "binary_mib"]
    )
    historical_frame = pd.DataFrame(historical_rows(historical_manifests)).sort_values(
        "language"
    )
    atomic_csv(candidate_frame, STAGING_RUN_DIR / "candidate_metrics.csv")
    atomic_csv(historical_frame, STAGING_RUN_DIR / "historical_metrics.csv")
    atomic_json(selection, STAGING_RUN_DIR / "selection.json")

    report = {
        "schema": 1,
        "cell": "20.K3",
        "status": "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING",
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "run_id": K3_RUN_ID,
        "contract_sha256": K3_CONTRACT_SHA256,
        "k2_run_id": EXPECTED_K2_RUN_ID,
        "k2_contract_sha256": EXPECTED_K2_CONTRACT_SHA256,
        "kenlm_commit": KENLM_COMMIT,
        "toolchain": toolchain,
        "heldout": HELDOUT_MANIFEST,
        "train_audits": train_audits,
        "candidate_manifests": candidate_manifests,
        "historical_manifests": historical_manifests,
        "selection": selection,
        "resource_gate_scope": (
            "LM seul; 20.K4/final doit mesurer BF16+adaptateur+un LM lazy "
            "avec marge 15% sous 8 Gio"
        ),
        "historical_models_modified": False,
        "active_runtime_modified": False,
        "submission_created": False,
        "reports_contain_raw_corpus_or_heldout_text": False,
        "next_step": (
            "20.K4 — comparer les finalistes V6 et les V4/V5 historiques "
            "sur les memes logits acoustiques, tuning puis audit WER separes"
        ),
    }
    atomic_json(report, STAGING_RUN_DIR / "kenlm_build_eval_20_K3.json")

    artifact_relatives = [
        "contract.json",
        "heldout_manifest.json",
        "toolchain.json",
        "candidate_metrics.csv",
        "historical_metrics.csv",
        "selection.json",
        "kenlm_build_eval_20_K3.json",
        *[
            f"manifests/{language}_{spec['name']}.json"
            for language in LANGUAGES
            for spec in CANDIDATE_GRID
        ],
        *[
            f"shortlist/{language}/unigrams.txt" for language in LANGUAGES
        ],
        *[
            f"shortlist/{language}/{finalist['candidate']}.binary"
            for language in LANGUAGES
            for finalist in selection[language]["v6_finalists"]
        ],
    ]
    artifacts = {
        relative: artifact_metadata(STAGING_RUN_DIR / relative)
        for relative in artifact_relatives
    }
    complete = {
        "schema": 1,
        "cell": "20.K3",
        "status": "PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING",
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "run_id": K3_RUN_ID,
        "contract_sha256": K3_CONTRACT_SHA256,
        "kenlm_commit": KENLM_COMMIT,
        "languages": list(LANGUAGES),
        "candidate_count": total_candidates,
        "artifacts": artifacts,
        "historical_models_modified": False,
        "active_runtime_modified": False,
        "submission_created": False,
    }
    atomic_json(complete, STAGING_RUN_DIR / "_COMPLETE.json")
    assert validate_completed_run(STAGING_RUN_DIR) is not None
    assert not FINAL_RUN_DIR.exists()
    os.replace(STAGING_RUN_DIR, FINAL_RUN_DIR)
    assert validate_completed_run(FINAL_RUN_DIR) is not None
    cleanup_unlisted_work_files(FINAL_RUN_DIR)
    assert validate_completed_run(FINAL_RUN_DIR) is not None
    repair_report_pointers()

    print("\n=== CANDIDATS V6 — MEILLEUR PAR LANGUE ===")
    print(
        candidate_frame.groupby("language", sort=False).head(1)[
            [
                "language",
                "candidate",
                "combined_ppl",
                "combined_oov_rate",
                "binary_mib",
                "query_rss_gib",
            ]
        ].to_string(index=False)
    )
    print("\n=== SHORTLIST 20.K4 ===")
    for language in LANGUAGES:
        names = [
            item["candidate"] for item in selection[language]["v6_finalists"]
        ]
        print(
            language,
            "-> V6",
            names,
            "+ historique",
            selection[language]["historical_baseline"]["family"],
        )
    print("STATUT 20.K3 : PASS_READY_FOR_20_K4_ACOUSTIC_WER_TUNING")
    print("Modeles       :", FINAL_RUN_DIR)
    print("Rapport       :", K3_REPORT_ROOT / K3_RUN_ID / "kenlm_build_eval_20_K3.json")
    print("Aucun runtime actif et aucune soumission n'ont ete modifies.")
    print("Etape suivante : 20.K4 — tuning WER acoustique V6 vs historiques")


main()
